In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset('starvector/text2svg-stack')

In [ ]:
dataset

In [ ]:
dataset['train']['Svg'][2], dataset['train']['caption_blip2'][2]

In [ ]:
dataset['train']['caption_llava'][2]

In [ ]:
from IPython.core.display import SVG


In [ ]:
display(SVG(dataset['train']['Svg'][2]))

# Starvector 1b

In [1]:
!pwd

/content


In [2]:
!git clone https://github.com/joanrod/star-vector

fatal: destination path 'star-vector' already exists and is not an empty directory.


In [1]:
cd star-vector

/content/star-vector


In [4]:
# !pip install .

In [9]:
!pip install git+https://github.com/huggingface/transformers.git svgpathtools cairosvg omegaconf fairscale -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
model_zoo = [
    'starvector/starvector-1b-im2svg',
    'starvector/starvector-8b-im2svg',
]

In [11]:
!pip install flash_attn -q

In [ ]:
import os
os.environ['HUGGINGFACEHUB_API_TOKEN'] = ''

In [31]:
from huggingface_hub import login
login(token=os.environ['HUGGINGFACEHUB_API_TOKEN'])

In [9]:
from PIL import Image
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoProcessor
from starvector.data.util import process_and_rasterize_svg
import torch

In [ ]:


model_name = "starvector/starvector-1b-im2svg"

starvector = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    attn_implementation=None
)
processor = starvector.model.processor
tokenizer = starvector.model.svg_transformer.tokenizer

starvector.cuda()
starvector.eval()



In [29]:
starvector.config.use_flash_attn = False

In [35]:
starvector.config

StarVectorConfig {
  "_attn_implementation_autoset": true,
  "adapter_norm": "batch_norm",
  "adapter_size": "large",
  "architectures": [
    "StarVectorForCausalLM"
  ],
  "auto_map": {
    "AutoConfig": "starvector/starvector-1b-im2svg--starvector_arch.StarVectorConfig",
    "AutoModelForCausalLM": "starvector/starvector-1b-im2svg--starvector_arch.StarVectorForCausalLM"
  },
  "dropout": 0.1,
  "hidden_size": 2048,
  "hidden_size_scale": 2,
  "image_encoder_type": "clip",
  "image_size": 224,
  "image_token_index": 49154,
  "init_type": "glorot",
  "max_length_train": 8192,
  "max_position_embeddings": 8192,
  "model_type": "starvector",
  "multi_query": true,
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "num_kv_heads": 4,
  "starcoder_model_name": "bigcode/starcoderbase-1b",
  "torch_dtype": "float16",
  "train_LLM": true,
  "train_image_encoder": false,
  "transformers_version": "4.50.0",
  "use_cache": true,
  "use_flash_attn": false,
  "vocab_size": 49156
}

In [37]:
image_pil = Image.open('assets/examples/sample-18.png')

image = processor(image_pil, return_tensors="pt")['pixel_values'].cuda()
# if not image.shape[0] == 1:
#     image = image.squeeze(0)

# Ensure image remains in batch format
if image.dim() == 3:  # If squeezed, re-add batch dimension
    image = image.unsqueeze(0)

batch = {"image": image}

raw_svg = starvector.generate_im2svg(batch, max_length=4000)[0]
svg, raster_image = process_and_rasterize_svg(raw_svg)

RuntimeError: FlashAttention only supports Ampere GPUs or newer.

In [10]:
!pip install -U bitsandbytes -q

In [8]:
import torch

In [5]:
from transformers import BitsAndBytesConfig

In [22]:
# Quantization Configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

In [30]:
from PIL import Image
from starvector.model.starvector_arch import StarVectorForCausalLM
from starvector.data.util import process_and_rasterize_svg

model_name = "starvector/starvector-1b-im2svg"

starvector = StarVectorForCausalLM.from_pretrained(
    model_name,
    # quantization_config=quantization_config,
    # device_map = 'auto'
)


# # Quantize the underlying transformer within StarCoderModel
# starvector.model.svg_transformer.transformer = starvector.model.svg_transformer.transformer.quantize(quantization_config)


starvector.cuda()
starvector.eval()



image_pil = Image.open('assets/examples/sample-0.png')
image = starvector.process_images([image_pil])[0].cuda().to(torch.float16)
batch = {"image": image}

raw_svg = starvector.generate_im2svg(batch, max_length=1000)[0]
svg, raster_image = process_and_rasterize_svg(raw_svg)

trainable params: 0 || all params: 1434095106 || trainable%: 0.0


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 386.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 18.12 MiB is free. Process 258276 has 14.72 GiB memory in use. Of the allocated memory 14.59 GiB is allocated by PyTorch, and 19.57 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)